# NovaX vs PyTorch — GPU Benchmark

> **Before running**: Go to **Runtime → Change runtime type → T4 GPU** (or better).

This notebook benchmarks NovaX against PyTorch on:
- Elementwise ops (add, mul, exp)
- Activation functions (relu, sigmoid, tanh, softmax)
- Reductions (sum, mean)
- Matrix multiplication (64×64 → 4096×4096)
- MLP forward pass
- MLP forward + backward
- **Kernel fusion** — NovaX's core structural advantage
- Fused matmul+bias+relu kernel
- Memory bandwidth
- Repeated inference throughput

In [ ]:
# ── 1. Install dependencies ───────────────────────────────────────────────
# PyCUDA (needs CUDA toolkit — present in Colab GPU runtimes)
!pip install pycuda -q

# NovaX from the development branch
!pip install 'git+https://github.com/SS-012/novax.git@claude/codebase-review-HJhA8#egg=novax[gpu]' -q

# PyTorch is pre-installed in Colab; upgrade only if needed
# !pip install torch --upgrade -q

print('Installation complete.')

In [ ]:
# ── 2. Verify GPU ─────────────────────────────────────────────────────────
import subprocess, sys

!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

import torch
print(f'PyTorch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**2:.0f} MB')

import novax as nx
print(f'NovaX {nx.__version__}  |  GPU available: {nx.GPU_AVAILABLE}')

if not (torch.cuda.is_available() and nx.GPU_AVAILABLE):
    sys.exit('GPU not available. Switch to a GPU runtime and re-run.')

In [ ]:
# ── 3. Benchmark helpers ──────────────────────────────────────────────────
import time, statistics, numpy as np
import pycuda.driver as cuda

WARMUP = 20
RUNS   = 100
torch_device = torch.device('cuda:0')

def time_novax(fn, warmup=WARMUP, runs=RUNS):
    for _ in range(warmup):
        fn()
    cuda.Context.synchronize()
    times = []
    for _ in range(runs):
        cuda.Context.synchronize()
        t0 = time.perf_counter()
        fn()
        cuda.Context.synchronize()
        times.append(time.perf_counter() - t0)
    return statistics.median(times) * 1_000  # ms

def time_torch(fn, warmup=WARMUP, runs=RUNS):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    times = []
    for _ in range(runs):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return statistics.median(times) * 1_000

results = {}  # store all results for the final summary

def ratio_str(r):
    if   r < 0.90: return f'{r:.2f}x  🚀 FASTER'
    elif r < 0.95: return f'{r:.2f}x  🟢 faster'
    elif r <= 1.05: return f'{r:.2f}x  ≈  TIED'
    elif r <= 1.20: return f'{r:.2f}x  🟡 slower'
    else:           return f'{r:.2f}x  🔴 SLOWER'

def bench(label, fn_nx, fn_pt, section='misc', **kwargs):
    nx_ms = time_novax(fn_nx, **kwargs)
    pt_ms = time_torch(fn_pt, **kwargs)
    r = nx_ms / pt_ms
    results.setdefault(section, []).append((label, nx_ms, pt_ms, r))
    print(f'{label:<45} NovaX {nx_ms:>7.3f}ms  PyTorch {pt_ms:>7.3f}ms  {ratio_str(r)}')

def nx_gpu(arr):
    t = nx.Tensor(arr.copy()); t.to_gpu(); return t

def pt_gpu(arr):
    return torch.from_numpy(arr.copy()).to(torch_device)

print('Helpers ready.')

In [ ]:
# ── 4. Elementwise ops ────────────────────────────────────────────────────
print('=== ELEMENTWISE OPS ===')
for sz in [10_000, 100_000, 1_000_000, 10_000_000]:
    arr = np.random.randn(sz).astype(np.float32)
    a_nx = nx_gpu(arr); b_nx = nx_gpu(arr + 1.0)
    a_pt = pt_gpu(arr); b_pt = pt_gpu(arr + 1.0)
    bench(f'add          n={sz:>10,}', lambda: nx.add(a_nx, b_nx).eval(), lambda: a_pt + b_pt,        section='elementwise')
    bench(f'mul          n={sz:>10,}', lambda: nx.mul(a_nx, b_nx).eval(), lambda: a_pt * b_pt,        section='elementwise')
    bench(f'exp          n={sz:>10,}', lambda: nx.exp(a_nx).eval(),       lambda: torch.exp(a_pt),    section='elementwise')

In [ ]:
# ── 5. Activations ────────────────────────────────────────────────────────
print('=== ACTIVATIONS (n=1,000,000) ===')
arr = np.random.randn(1_000_000).astype(np.float32)
a_nx = nx_gpu(arr); a_pt = pt_gpu(arr)
bench('relu',    lambda: nx.relu(a_nx).eval(),    lambda: torch.relu(a_pt),            section='activations')
bench('sigmoid', lambda: nx.sigmoid(a_nx).eval(), lambda: torch.sigmoid(a_pt),         section='activations')
bench('tanh',    lambda: nx.tanh(a_nx).eval(),    lambda: torch.tanh(a_pt),             section='activations')
arr_s = np.random.randn(50_000).astype(np.float32)
a_s_nx = nx_gpu(arr_s); a_s_pt = pt_gpu(arr_s)
bench('softmax (n=50,000)', lambda: nx.softmax(a_s_nx).eval(), lambda: torch.softmax(a_s_pt, dim=0), section='activations')

In [ ]:
# ── 6. Reductions ─────────────────────────────────────────────────────────
print('=== REDUCTIONS ===')
for sz in [100_000, 1_000_000, 10_000_000]:
    arr = np.random.randn(sz).astype(np.float32)
    a_nx = nx_gpu(arr); a_pt = pt_gpu(arr)
    bench(f'sum   n={sz:>10,}', lambda: nx.sum(a_nx).eval(),  lambda: a_pt.sum(),  section='reductions')
    bench(f'mean  n={sz:>10,}', lambda: nx.mean(a_nx).eval(), lambda: a_pt.mean(), section='reductions')

In [ ]:
# ── 7. Matmul ─────────────────────────────────────────────────────────────
print('=== MATRIX MULTIPLICATION ===')
for M, K, N in [(64,64,64),(256,256,256),(512,512,512),(1024,1024,1024),(2048,2048,2048),(4096,4096,4096)]:
    A = np.random.randn(M, K).astype(np.float32)
    B = np.random.randn(K, N).astype(np.float32)
    A_nx = nx_gpu(A); B_nx = nx_gpu(B)
    A_pt = pt_gpu(A); B_pt = pt_gpu(B)
    bench(f'matmul ({M}×{K}) @ ({K}×{N})',
          lambda: nx.matmul(A_nx, B_nx).eval(), lambda: A_pt @ B_pt, section='matmul')

In [ ]:
# ── 8. MLP forward ────────────────────────────────────────────────────────
print('=== MLP FORWARD (batch=128) ===')
BS = 128
for IN, HID, OUT in [(128,256,128),(256,512,256),(512,1024,512)]:
    x_a  = np.random.randn(BS, IN).astype(np.float32)
    W1_a = np.random.randn(IN, HID).astype(np.float32) * 0.02
    b1_a = np.zeros(HID, dtype=np.float32)
    W2_a = np.random.randn(HID, OUT).astype(np.float32) * 0.02
    b2_a = np.zeros(OUT, dtype=np.float32)
    X_nx=nx_gpu(x_a);  W1_nx=nx_gpu(W1_a); b1_nx=nx_gpu(b1_a)
    W2_nx=nx_gpu(W2_a); b2_nx=nx_gpu(b2_a)
    X_pt=pt_gpu(x_a);  W1_pt=pt_gpu(W1_a); b1_pt=pt_gpu(b1_a)
    W2_pt=pt_gpu(W2_a); b2_pt=pt_gpu(b2_a)
    def nx_fwd():
        h = nx.relu(nx.matmul(X_nx, W1_nx) + b1_nx)
        return nx.mean(nx.matmul(h, W2_nx) + b2_nx).eval()
    def pt_fwd():
        h = torch.relu(X_pt @ W1_pt + b1_pt)
        return (h @ W2_pt + b2_pt).mean()
    bench(f'fwd in={IN} hid={HID} out={OUT}', nx_fwd, pt_fwd, section='mlp_fwd')

In [ ]:
# ── 9. MLP forward + backward ─────────────────────────────────────────────
print('=== MLP FORWARD + BACKWARD (batch=128, 256→128→64) ===')
IN, HID, OUT, BS = 256, 128, 64, 128
x_a  = np.random.randn(BS, IN).astype(np.float32)
W1_a = np.random.randn(IN, HID).astype(np.float32) * 0.02
b1_a = np.zeros(HID, dtype=np.float32)
W2_a = np.random.randn(HID, OUT).astype(np.float32) * 0.02
b2_a = np.zeros(OUT, dtype=np.float32)

X_nx = nx_gpu(x_a)
W1_nx = nx.Tensor(W1_a.copy(), requires_grad=True); W1_nx.to_gpu()
b1_nx = nx.Tensor(b1_a.copy(), requires_grad=True); b1_nx.to_gpu()
W2_nx = nx.Tensor(W2_a.copy(), requires_grad=True); W2_nx.to_gpu()
b2_nx = nx.Tensor(b2_a.copy(), requires_grad=True); b2_nx.to_gpu()

X_pt  = pt_gpu(x_a)
W1_pt = torch.tensor(W1_a.copy(), device=torch_device, requires_grad=True)
b1_pt = torch.tensor(b1_a.copy(), device=torch_device, requires_grad=True)
W2_pt = torch.tensor(W2_a.copy(), device=torch_device, requires_grad=True)
b2_pt = torch.tensor(b2_a.copy(), device=torch_device, requires_grad=True)

def nx_fwd_bwd():
    for p in [W1_nx, b1_nx, W2_nx, b2_nx]: p.grad = None
    h    = nx.relu(nx.matmul(X_nx, W1_nx) + b1_nx)
    loss = nx.mean(nx.matmul(h, W2_nx) + b2_nx)
    loss.eval().backward()

def pt_fwd_bwd():
    for p in [W1_pt, b1_pt, W2_pt, b2_pt]: p.grad = None
    h    = torch.relu(X_pt @ W1_pt + b1_pt)
    loss = (h @ W2_pt + b2_pt).mean()
    loss.backward()

bench('forward + backward', nx_fwd_bwd, pt_fwd_bwd, section='mlp_bwd')

In [ ]:
# ── 10. KERNEL FUSION — NovaX's core advantage ────────────────────────────
# NovaX compiles elementwise chains into ONE CUDA kernel.
# Stock PyTorch (without torch.compile) launches one kernel per op.
# This section shows where fusion matters most.
print('=== KERNEL FUSION vs PyTorch (n=1,000,000) ===')
arr = np.random.randn(1_000_000).astype(np.float32)
a_nx = nx_gpu(arr); b_nx = nx_gpu(arr * 0.5); c_nx = nx_gpu(arr + 1.0)
a_pt = pt_gpu(arr); b_pt = pt_gpu(arr * 0.5); c_pt = pt_gpu(arr + 1.0)

# 2-op: a + b
bench('a + b            [2 ops → 1 kernel (NovaX)]',
      lambda: nx.add(a_nx, b_nx).eval(), lambda: a_pt + b_pt, section='fusion')

# 3-op: relu(a * b + c)
bench('relu(a*b + c)     [3 ops → 1 kernel (NovaX)]',
      lambda: nx.relu(nx.mul(a_nx, b_nx) + c_nx).eval(),
      lambda: torch.relu(a_pt * b_pt + c_pt), section='fusion')

# 5-op: sigmoid(relu(a*b+c) * a)
bench('sigmoid(relu(a*b+c)*a)  [5 ops → 1 kernel (NovaX)]',
      lambda: nx.sigmoid(nx.relu(nx.mul(a_nx, b_nx) + c_nx) * a_nx).eval(),
      lambda: torch.sigmoid(torch.relu(a_pt * b_pt + c_pt) * a_pt), section='fusion')

# 6-op: tanh(exp(a) + relu(b * c))
bench('tanh(exp(a) + relu(b*c))  [6 ops → 1 kernel (NovaX)]',
      lambda: nx.tanh(nx.exp(a_nx) + nx.relu(nx.mul(b_nx, c_nx))).eval(),
      lambda: torch.tanh(torch.exp(a_pt) + torch.relu(b_pt * c_pt)), section='fusion')

print()
print('NOTE: PyTorch torch.compile() can also fuse kernels.')
print('      The comparison below shows NovaX vs torch.compile:')

# torch.compile version (if PyTorch >= 2.0)
try:
    def pt_chain3(a, b, c): return torch.relu(a * b + c)
    pt_chain3_compiled = torch.compile(pt_chain3)
    # Warm up compile
    _ = pt_chain3_compiled(a_pt, b_pt, c_pt)
    torch.cuda.synchronize()
    bench('relu(a*b+c) vs torch.compile  [3 ops]',
          lambda: nx.relu(nx.mul(a_nx, b_nx) + c_nx).eval(),
          lambda: pt_chain3_compiled(a_pt, b_pt, c_pt), section='fusion_compile')
except Exception as e:
    print(f'torch.compile not available: {e}')

In [ ]:
# ── 11. Fused matmul+bias+relu ────────────────────────────────────────────
print('=== FUSED matmul+bias+relu ===')
for M, K, N in [(128,256,128),(256,512,256),(512,1024,512)]:
    X_a = np.random.randn(M, K).astype(np.float32)
    W_a = np.random.randn(K, N).astype(np.float32)
    b_a = np.zeros(N, dtype=np.float32)
    X_nx=nx_gpu(X_a); W_nx=nx_gpu(W_a); b_nx_f=nx_gpu(b_a)
    X_pt=pt_gpu(X_a); W_pt=pt_gpu(W_a); b_pt_f=pt_gpu(b_a)
    bench(f'({M}×{K})@({K}×{N})+b+relu  fused vs naive',
          lambda: nx.launch_matmul_bias_relu(X_nx, W_nx, b_nx_f),
          lambda: torch.relu(X_pt @ W_pt + b_pt_f), section='fused_mm')
    bench(f'({M}×{K})@({K}×{N})+b+relu  fused vs nn.linear',
          lambda: nx.launch_matmul_bias_relu(X_nx, W_nx, b_nx_f),
          lambda: torch.relu(torch.nn.functional.linear(X_pt, W_pt.T, b_pt_f)), section='fused_mm')

In [ ]:
# ── 12. Memory bandwidth ──────────────────────────────────────────────────
print('=== MEMORY BANDWIDTH (n=10,000,000) ===')
arr = np.random.randn(10_000_000).astype(np.float32)
a_nx=nx_gpu(arr); b_nx=nx_gpu(arr*0.5)
a_pt=pt_gpu(arr); b_pt=pt_gpu(arr*0.5)
bench('add (reads 2 arrays, writes 1)',
      lambda: nx.add(a_nx, b_nx).eval(), lambda: a_pt + b_pt, section='bandwidth')
bench('neg (reads 1 array, writes 1)',
      lambda: nx.neg(a_nx).eval(), lambda: -a_pt, section='bandwidth')
bench('sqrt(|x|)  (2 ops)',
      lambda: nx.sqrt(nx.abs(a_nx)).eval(), lambda: torch.sqrt(torch.abs(a_pt)), section='bandwidth')

# Effective GB/s
mb = arr.nbytes / 1024**2
nx_t = time_novax(lambda: nx.add(a_nx, b_nx).eval()) / 1000
pt_t = time_torch(lambda: a_pt + b_pt) / 1000
print(f'\nEffective bandwidth (add: 3 arrays × {mb:.0f} MB):')
print(f'  NovaX  : {3*mb/nx_t/1024:.1f} GB/s')
print(f'  PyTorch: {3*mb/pt_t/1024:.1f} GB/s')

In [ ]:
# ── 13. Repeated inference throughput ─────────────────────────
print('=== REPEATED INFERENCE THROUGHPUT (1000 passes, batch=64) ===')
IN, HID, OUT, BS = 128, 256, 128, 64
x_a  = np.random.randn(BS, IN).astype(np.float32)
W1_a = np.random.randn(IN, HID).astype(np.float32) * 0.02
b1_a = np.zeros(HID, dtype=np.float32)
W2_a = np.random.randn(HID, OUT).astype(np.float32) * 0.02
b2_a = np.zeros(OUT, dtype=np.float32)
X_nx=nx_gpu(x_a); W1_nx=nx_gpu(W1_a); b1_nx=nx_gpu(b1_a); W2_nx=nx_gpu(W2_a); b2_nx=nx_gpu(b2_a)
X_pt=pt_gpu(x_a); W1_pt=pt_gpu(W1_a); b1_pt=pt_gpu(b1_a); W2_pt=pt_gpu(W2_a); b2_pt=pt_gpu(b2_a)

def nx_inf():
    h = nx.relu(nx.matmul(X_nx, W1_nx) + b1_nx)
    return (nx.matmul(h, W2_nx) + b2_nx).eval()

def pt_inf():
    with torch.no_grad():
        h = torch.relu(X_pt @ W1_pt + b1_pt)
        return h @ W2_pt + b2_pt

N = 1000
cuda.Context.synchronize()
t0 = time.perf_counter()
for _ in range(N): nx_inf()
cuda.Context.synchronize()
nx_total = (time.perf_counter() - t0) * 1000

torch.cuda.synchronize()
t0 = time.perf_counter()
for _ in range(N): pt_inf()
torch.cuda.synchronize()
pt_total = (time.perf_counter() - t0) * 1000

r = (nx_total/N) / (pt_total/N)
print(f'eager   : NovaX {nx_total/N:.3f}ms  PyTorch {pt_total/N:.3f}ms  {ratio_str(r)}')
print(f'Throughput: NovaX {N/(nx_total/1000):.0f} passes/s  |  PyTorch {N/(pt_total/1000):.0f} passes/s')

# Captured replay: build the graph once, then replay the whole forward pass
# with zero per-op Python. This is NovaX's structural advantage over eager
# dispatch (compared here against PyTorch eager — no torch.compile / CUDA graphs).
graph = nx.CUDAGraph()
graph.capture(nx_inf)                # warm caches + capture (falls back if unsupported)
for _ in range(20): graph.replay()   # warmup
cuda.Context.synchronize()
t0 = time.perf_counter()
for _ in range(N): graph.replay()
cuda.Context.synchronize()
nx_cap = (time.perf_counter() - t0) * 1000

rc = (nx_cap/N) / (pt_total/N)
print(f'captured: NovaX {nx_cap/N:.3f}ms  PyTorch {pt_total/N:.3f}ms  {ratio_str(rc)}')
print(f'Throughput: NovaX {N/(nx_cap/1000):.0f} passes/s (captured)  |  PyTorch {N/(pt_total/1000):.0f} passes/s (eager)')
print(f'Capture/replay speedup vs NovaX eager: {nx_total/nx_cap:.2f}x')

In [ ]:
# ── 14. Summary table + score ─────────────────────────────────────────────
import pandas as pd

rows = []
for section, entries in results.items():
    for label, nx_ms, pt_ms, r in entries:
        rows.append({'Section': section, 'Operation': label,
                     'NovaX (ms)': round(nx_ms, 4),
                     'PyTorch (ms)': round(pt_ms, 4),
                     'Ratio': round(r, 3),
                     'Winner': '🚀 NovaX' if r < 0.95 else ('≈ Tied' if r <= 1.05 else '🔴 PyTorch')})

df = pd.DataFrame(rows)
wins   = (df['Winner'] == '🚀 NovaX').sum()
ties   = (df['Winner'] == '≈ Tied').sum()
losses = (df['Winner'] == '🔴 PyTorch').sum()
total  = len(df)

print('\n' + '='*60)
print(f'  SCOREBOARD: NovaX {wins}W / {ties}T / {losses}L  ({total} benchmarks)')
print('='*60)
print(f'  🚀 NovaX faster:   {wins:3d} ({100*wins/total:.0f}%)')
print(f'  ≈  Tied:           {ties:3d} ({100*ties/total:.0f}%)')
print(f'  🔴 PyTorch faster: {losses:3d} ({100*losses/total:.0f}%)')
print('='*60)

# Show full DataFrame
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 50)
df.style.background_gradient(subset=['Ratio'], cmap='RdYlGn_r')

In [ ]:
# ── 15. Visualisation ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(3, 2, figsize=(16, 14))
fig.suptitle('NovaX vs PyTorch — GPU Benchmark', fontsize=16, fontweight='bold', y=1.01)

section_titles = {
    'elementwise': 'Elementwise Ops',
    'activations': 'Activations',
    'reductions':  'Reductions',
    'matmul':      'Matrix Multiplication',
    'fusion':      'Kernel Fusion',
    'mlp_fwd':     'MLP Forward Pass',
}

for ax, (sec, title) in zip(axes.flat, section_titles.items()):
    if sec not in results: ax.set_visible(False); continue
    entries = results[sec]
    labels  = [e[0].split('n=')[0].strip()[:30] for e in entries]
    nx_vals = [e[1] for e in entries]
    pt_vals = [e[2] for e in entries]
    x = range(len(labels))
    w = 0.35
    bars_nx = ax.bar([i - w/2 for i in x], nx_vals, w, label='NovaX', color='#7C3AED', alpha=0.85)
    bars_pt = ax.bar([i + w/2 for i in x], pt_vals, w, label='PyTorch', color='#EE4C2C', alpha=0.85)
    ax.set_xticks(list(x))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Median time (ms)')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('novax_gpu_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as novax_gpu_benchmark.png')